# VideoMind v2 — Colab GPU Worker

Use this notebook to run the **heavy video processing pipeline** (Whisper transcription, frame extraction, CLIP visual embeddings, text embeddings, Pinecone indexing) on a free Colab GPU, while you develop and edit the FastAPI + Streamlit app locally in **VS Code**.

Both environments talk to the **same MongoDB Atlas** and **same Pinecone** project, so a video processed here immediately shows up as `ready` in your locally-running Streamlit app.

**Workflow:**
1. Write/edit code locally in VS Code and push to GitHub.
2. Run this notebook in Colab with a GPU runtime (`Runtime -> Change runtime type -> GPU`).
3. It clones your repo, installs dependencies, and lets you upload/process a video using the GPU.
4. Your local Streamlit app (pointed at the same MongoDB/Pinecone) reflects the processed video immediately.

## 1. Check GPU

In [1]:
!nvidia-smi

Fri Aug 21 04:41:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone your repo
Replace with your actual GitHub URL after you push this project.

In [2]:
REPO_URL = "https://github.com/anitkumarmaity1-hash/VideoMind.git"  # <-- EDIT ME

!git clone $REPO_URL
%cd VideoMind/backend

fatal: destination path 'VideoMind' already exists and is not an empty directory.
/content/VideoMind/backend


## 3. Install system + Python dependencies
(ffmpeg, then the backend's requirements.txt — same file used locally, so behavior stays identical)

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg

# IMPORTANT: do NOT install numpy, opencv, or torch here.
# This Colab image already ships mutually-compatible GPU-enabled torch,
# numpy (2.x), and a working opencv. Reinstalling them from our local
# requirements.txt (which pins older versions for local/CPU dev) creates
# a numpy ABI mismatch that causes 'AttributeError: module cv2 has no
# attribute VideoCapture' and/or 'RecursionError: maximum recursion
# depth exceeded' deep inside numpy's dtype printing code. Only install
# the packages this app needs on TOP of what Colab already provides.
!pip install -q \
    fastapi==0.115.0 "uvicorn[standard]==0.30.6" pydantic==2.9.2 pydantic-settings==2.5.2 \
    motor==3.6.0 "pymongo==4.9.1" certifi \
    pinecone-client==5.0.1 groq==0.11.0 \
    faster-whisper==1.0.3 open-clip-torch==2.26.1 sentence-transformers==3.1.1 \
    ffmpeg-python==0.2.0 python-multipart==0.0.9 python-dotenv==1.0.1 yt-dlp==2024.10.7

# Sanity check: confirm numpy/opencv/torch were NOT changed and are still consistent.
import numpy, cv2, torch
print('numpy:', numpy.__version__)
print('opencv:', cv2.__version__)
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
# quick smoke test that previously triggered the RecursionError/AttributeError
assert hasattr(cv2, 'VideoCapture'), "cv2.VideoCapture missing — opencv install is broken"
print(numpy.dtype('float32'))
print('Environment OK.')

## 4. Configure environment variables
Use Colab's "Secrets" (key icon in the left sidebar) to store these securely instead of pasting them in plain text, then load them here. Fill in the same values as your local `backend/.env`.

In [9]:
import os

# Option A: Colab secrets (recommended)
from google.colab import userdata
os.environ['MONGO_URI'] = userdata.get('MONGO_URI')
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# Option B: paste directly (fine for quick testing, do NOT commit this notebook with real keys filled in)
# os.environ['MONGO_URI'] = 'mongodb+srv://<user>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority'
os.environ['MONGO_DB_NAME'] = '24mca00py0055_db_user'
# os.environ['PINECONE_API_KEY'] = '<your-pinecone-api-key>'
os.environ['PINECONE_ENV'] = 'us-east-1'
os.environ['PINECONE_TEXT_INDEX'] = 'videomind-text'
os.environ['PINECONE_VISUAL_INDEX'] = 'videomind-visual'
os.environ['TEXT_EMBEDDING_DIM'] = '384'
os.environ['VISUAL_EMBEDDING_DIM'] = '512'
# os.environ['GROQ_API_KEY'] = '<your-groq-api-key>'
os.environ['GROQ_MODEL'] = 'openai/gpt-oss-20b'
os.environ['WHISPER_MODEL'] = 'small'
os.environ['WHISPER_DEVICE'] = 'cuda'          # GPU!
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16' # faster on GPU
os.environ['TEXT_EMBEDDING_MODEL'] = 'BAAI/bge-small-en-v1.5'
os.environ['VISUAL_EMBEDDING_MODEL'] = 'ViT-B-32'
os.environ['VISUAL_EMBEDDING_PRETRAINED'] = 'laion2b_s34b_b79k'
os.environ['STORAGE_BACKEND'] = 'local'
os.environ['LOCAL_DATA_DIR'] = './data'
os.environ['MAX_UPLOAD_SIZE_MB'] = '500'
os.environ['ALLOWED_VIDEO_EXTENSIONS'] = '.mp4,.mov,.mkv'
print('Environment configured.')

Environment configured.


## 5. Upload a video (Colab file picker)

In [10]:
from google.colab import files
uploaded = files.upload()
local_filename = list(uploaded.keys())[0]
print(f'Uploaded: {local_filename}')

Saving vidssave.com How to Get a REMOTE JOB With Zero Experience 480P.mp4 to vidssave.com How to Get a REMOTE JOB With Zero Experience 480P.mp4
Uploaded: vidssave.com How to Get a REMOTE JOB With Zero Experience 480P.mp4


## 6. Register the video in MongoDB and run the pipeline
This calls the exact same `run_pipeline()` function used by the FastAPI backend locally — no duplicated logic, so results are identical either way.

In [11]:
import sys, os, shutil, asyncio
sys.path.insert(0, os.getcwd())  # backend/ is already the cwd

from app.database.mongo import videos_collection, ensure_indexes
from app.models.video import VideoMetadata
from app.utils.validation import generate_video_id
from app.pipeline.pipeline_runner import run_pipeline

async def process_uploaded_video(local_path: str):
    await ensure_indexes()

    video_id = generate_video_id()
    ext = os.path.splitext(local_path)[1]
    dest_dir = os.path.join('data', 'videos')
    os.makedirs(dest_dir, exist_ok=True)
    dest_path = os.path.join(dest_dir, f'{video_id}{ext}')
    shutil.copy(local_path, dest_path)

    doc = VideoMetadata(video_id=video_id, filename=local_path, storage_path=dest_path, source='upload')
    await videos_collection().insert_one(doc.model_dump())

    print(f'Registered video_id={video_id}. Running pipeline on GPU...')
    await run_pipeline(video_id)
    print(f'Done! video_id={video_id} is now READY.')
    return video_id

video_id = await process_uploaded_video(local_filename)
print('\nUse this video_id in your local Streamlit app:', video_id)

Registered video_id=vid_9061099b06b5. Running pipeline on GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'TypeError: Failed to fetch'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


RecursionError: maximum recursion depth exceeded

## 7. (Optional) Ask a question right here to sanity-check retrieval before switching back to the UI

In [ ]:
from app.pipeline.retrieval import retrieve_text, retrieve_visual, fuse_scores
from app.services.llm_service import generate_grounded_answer
from app.utils.timestamps import format_timestamp

question = 'Summarize what this video is about'
text_results = retrieve_text(question, video_id)
visual_results = retrieve_visual(question, video_id)
fused = fuse_scores(text_results, visual_results)

text_evidence = [{
    'start_formatted': format_timestamp(r['metadata']['start_time']),
    'end_formatted': format_timestamp(r['metadata']['end_time']),
    'content': r['metadata'].get('transcript', ''),
} for r in text_results]

answer = generate_grounded_answer(question, text_evidence, [], answer_mode='standard')
print(answer)

## Next step

Go back to your local VS Code environment, start the backend + frontend (see `README.md`), and either:
- type the printed `video_id` into the Streamlit sidebar's "Or load existing video_id" field, or
- process a new video directly from the local UI (slower on CPU, but works).